# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_05 — Traducción automática con `ai.translate`

**Serie:** Microsoft Fabric AI Functions con Pandas

**Modelo:** `gpt-5-mini`

Objetivos: traducir textos multilingües, validar nulos y terminología, medir tiempo, tokens y CU-segundos. Requisitos: capacidad Fabric compatible, permisos de tenant para AI Functions y runtime Python compatible. Consultar la configuración oficial antes de instalar dependencias.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, dff48d0e-c368-4253-91c8-c1f941e2007e, 3, Finished, Available, Finished, False)

## 2. Configuración
Modelo gpt-5-mini, concurrencia de laboratorio 10 y barra de progreso con estadísticas. [Configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration).

In [2]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "gpt-5-mini"
CONCURRENCY = 10
aifunc.default_conf.model_deployment_name = MODEL_NAME
aifunc.default_conf.reasoning_effort = "low"
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, dff48d0e-c368-4253-91c8-c1f941e2007e, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.translate con pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/translate)

```python
df["translations"] = df["text"].ai.translate("spanish")
```

| Parámetro | Tipo | Uso |
|---|---|---|
| `to_lang` | str, obligatorio | Idioma de destino. |
| `column_type` | str, opcional | `path` para rutas de archivos multimodales. |

Retorna una Series de traducciones por fila; una entrada nula devuelve nulo. No requiere indicar idioma de origen. Microsoft enumera diez idiomas probados: checo, inglés, finés, francés, alemán, griego, italiano, polaco, español y sueco. Otros idiomas, como portugués, pueden presentar resultados variables. Los ejercicios siguientes son propios.

## 4. Dataset ficticio
Mismos cinco incidentes del NB_05 PySpark, con una entrada nula.

In [3]:
data = [(1, "The payment service is unavailable."),
        (2, "Le tableau de bord ne se met pas à jour."),
        (3, "O acesso ao sistema foi restaurado."),
        (4, "The pipeline failed after the schema change."),
        (5, None)]
df = pd.DataFrame(data, columns=["id_incidente", "texto"])
display(df)

StatementMeta(, dff48d0e-c368-4253-91c8-c1f941e2007e, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fb9c8227-b884-4f84-b632-aa2921ba5ce2)

## 5. Traducción al español
Conservar la Series y sus estadísticas antes de asignarla al DataFrame. Medimos el tiempo de la llamada y memoria aproximada de los objetos.

In [4]:
runs = []
def ejecutar(nombre, entrada, **kwargs):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc).isoformat()
    input_bytes = int(entrada.memory_usage(deep=True).sum() if isinstance(entrada, pd.DataFrame) else entrada.memory_usage(deep=True))
    t0 = perf_counter()
    resultado = entrada.ai.translate(**kwargs)
    elapsed = perf_counter() - t0
    raw = resultado.ai.stats
    runs.append((nombre, run_id, started, elapsed, raw, resultado, len(entrada), input_bytes))
    return resultado

traduccion = ejecutar("spanish", df["texto"], to_lang="spanish")
display(df.assign(traduccion=traduccion))

StatementMeta(, dff48d0e-c368-4253-91c8-c1f941e2007e, 6, Finished, Available, Finished, False)

ai.translate: 100%|██████████| 5/5 [00:04<00:00,  1.11it/s, cached=0, in=753, out=379, CU-s=26.11]                             


SynapseWidget(Synapse.DataFrame, d6cd5bce-79d8-4c80-afa5-067f261f5801)

## 6. Traducción al portugués
Segunda llamada independiente sobre los mismos datos. A diferencia del ejercicio PySpark opcional, aquí se habilita para comparar y auditar ambos destinos. Consume capacidad adicional.

In [5]:
traduccion_pt = ejecutar("portuguese", df["texto"], to_lang="portuguese")
display(df.assign(traduccion_pt=traduccion_pt))

StatementMeta(, dff48d0e-c368-4253-91c8-c1f941e2007e, 7, Finished, Available, Finished, False)

ai.translate: 100%|██████████| 5/5 [00:02<00:00,  1.99it/s, cached=0, in=761, out=362, CU-s=24.98]                             


SynapseWidget(Synapse.DataFrame, 583b8d41-4c68-464f-bc0c-cf1c5ce7bf2a)

## 7. Validación y comparación
Distinguir entradas nulas esperadas y resultados faltantes en textos presentes. Revisar manualmente fidelidad y terminología técnica; la longitud y la presencia de texto no garantizan calidad.

In [6]:
resultado = df.assign(traduccion=traduccion, traduccion_pt=traduccion_pt)
for col in ("traduccion", "traduccion_pt"):
    estado = pd.Series("TRADUCIDO", index=df.index)
    estado.loc[resultado[col].isna() & df.texto.notna()] = "SIN_RESULTADO"
    estado.loc[df.texto.isna()] = "ENTRADA_NULA"
    resultado["estado_" + col] = estado
    display(estado.value_counts(dropna=False))
    print("Salidas inesperadas para entradas nulas:", int((df.texto.isna() & resultado[col].notna()).sum()))
display(resultado)

StatementMeta(, dff48d0e-c368-4253-91c8-c1f941e2007e, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da93d57c-ea74-4d09-8211-7546d37a6a07)

Salidas inesperadas para entradas nulas: 0


SynapseWidget(Synapse.DataFrame, eb4323ad-240c-400b-af0f-6ac7f1491afe)

Salidas inesperadas para entradas nulas: 0


SynapseWidget(Synapse.DataFrame, b32aed58-3b07-47e2-bec7-a0c5085b9254)

## 8. Estadísticas y tokens
Consultar cada ejecución por separado. No reemplazar métricas ausentes por cero.

In [7]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")
for nombre, _, _, _, raw, _, _, _ in runs:
    print(nombre)
    display(raw)

StatementMeta(, dff48d0e-c368-4253-91c8-c1f941e2007e, 9, Finished, Available, Finished, False)

spanish


SynapseWidget(Synapse.DataFrame, 4d8239a2-ff2b-489d-8e16-09cd956f2aef)

portuguese


SynapseWidget(Synapse.DataFrame, e1ad0430-c1cc-4ced-b6bd-025dce41dc8b)

## 9. Estimación de CU-segundos
Tarifas de referencia de la serie para gpt-5-mini integrado, en CU-s por 1.000 tokens:

| Entrada sin caché | Entrada en caché | Salida |
|---:|---:|---:|
| 8,40 | 0,84 | 67,23 |

$CU_s=((I-C)\times8.40+C\times0.84+O\times67.23)/1000$

C es un subconjunto de I. Los tokens de razonamiento no se suman nuevamente a O. Ejemplo hipotético: I=1000, C=200, O=100 da 13,611 CU-s. No representa una ejecución real ni el consumo total del runtime. Verificar vigencia en [facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing). Si faltan métricas o el modelo/cliente no coincide, no estimar.

## 10. Auditoría individual y acumulada
Registrar ambos destinos, tiempo y memoria aproximada. Los textos nulos esperados se excluyen del objetivo de éxito; las métricas ausentes permanecen nulas.

In [ ]:
records = []
for nombre, run_id, started, elapsed, raw, resultado_run, num_rows, input_bytes in runs:
    stats = normalizar(raw)
    def metric(k):
        v = stats.get(k)
        return None if v is None or pd.isna(v) else int(v)
    I,C,O = [metric(k) for k in ("input_tokens","cached_tokens","output_tokens")]
    cu = None
    if (stats.get("model") in (MODEL_NAME,"gpt-5-mini-2025-08-07") and stats.get("client_type")=="fabric_llm_endpoint" and all(v is not None for v in (I,C,O)) and 0<=C<=I and O>=0):
        cu = ((I-C)*8.40+C*0.84+O*67.23)/1000
    audit = dict(run_id=run_id, execution_timestamp_utc=started, notebook_name="NB_05_AI_TRANSLATE_PANDAS", operation="ai.translate:"+nombre, model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=num_rows, client_type=stats.get("client_type"), estimated_cu_seconds=cu, elapsed_seconds=elapsed, input_bytes=input_bytes, output_bytes=int(resultado_run.memory_usage(deep=True)), concurrency=CONCURRENCY)
    for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
        audit[key]=metric(key)
    expected = int(df.texto.notna().sum())
    if (audit["num_exceptions"] or 0) > 0 or (audit["num_harmful"] or 0) > 0:
        audit["execution_status"] = "PARTIAL"
    elif audit["num_successful"] == expected and audit["num_exceptions"] == 0 and audit["num_harmful"] == 0 and resultado_run[df.texto.notna()].notna().all():
        audit["execution_status"] = "SUCCESS"
    else:
        audit["execution_status"] = "INCOMPLETE"
    records.append(audit)
audit_df=pd.DataFrame(records)
display(audit_df)
print("CU-s acumulados:", audit_df.estimated_cu_seconds.sum(min_count=len(audit_df)))

## 11. Persistencia opcional en Delta
Requiere Lakehouse adjunto, deltalake y pyarrow. Persiste en Files, no registra una tabla SQL. Deshabilitada por defecto.

In [ ]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

## 12. Multimodalidad (opcional)
Usar archivos autorizados y revisar [requisitos multimodales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). Consumo independiente, fuera de la auditoría principal.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    archivos = pd.Series(["/lakehouse/default/Files/ejemplo.pdf"])
    resultado_archivos = archivos.ai.translate("spanish", column_type="path")
    display(resultado_archivos)
    display(resultado_archivos.ai.stats)

## 13. Comparación y conclusiones
En Pandas, `df["texto"].ai.translate("spanish")` devuelve una Series; en PySpark, `df.ai.translate(to_lang="spanish", input_col="texto", output_col="traduccion")` devuelve un DataFrame distribuido. La concurrencia de solicitudes no equivale al paralelismo Spark. Comparar la calidad manualmente y utilizar las métricas de auditoría para el futuro benchmarking.

Referencias: [método oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/translate), [configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration), [facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing).